# Intent Classifier V4 Training - Semantic Learning

This notebook trains on the **V4 dataset** designed for TRUE semantic learning.

## Key Validation Points:
- Epoch 1 accuracy should be **50-55%** (not 84%)
- Final accuracy should be **85-90%** (not 98%)
- If accuracy > 95%, the dataset still has issues

## Expected Training Curve:
| Epoch | V3 | V4 (Target) |
|-------|----|--------------|
| 1 | 84% | 50-55% |
| 2 | 98% | 60-65% |
| 5 | 99% | 80-85% |
| Final | 98% | 85-90% |

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset

# Project paths
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
MODEL_DIR = PROJECT_ROOT / "models" / "intent_classifier_v4"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Load V4 Dataset

print("Loading V4 dataset...")
print("=" * 60)

# Load train dataset
train_val_df = pd.read_csv(DATA_DIR / "intent_dataset_v4.csv")

# Load OOD test dataset
test_ood_df = pd.read_csv(DATA_DIR / "intent_dataset_v4_test_ood.csv")

# Clean data - only drop rows where text or intent is NaN
train_val_df = train_val_df.dropna(subset=['text', 'intent'])
train_val_df = train_val_df[train_val_df['text'].str.len() > 0]
test_ood_df = test_ood_df.dropna(subset=['text', 'intent'])
test_ood_df = test_ood_df[test_ood_df['text'].str.len() > 0]

print(f"Train+Val samples: {len(train_val_df)}")
print(f"Test OOD samples: {len(test_ood_df)}")

print(f"\nIntent distribution (Train+Val):")
print(train_val_df['intent'].value_counts())

print(f"\nSample type distribution:")
print(train_val_df['sample_type'].value_counts())

In [ ]:
# Cell 3: Load V4 Metadata

metadata_path = DATA_DIR / "dataset_v4_metadata.json"
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    print("V4 Dataset Metadata:")
    print("-" * 40)
    print(f"Version: {metadata.get('version')}")
    print(f"Description: {metadata.get('description')}")
    print(f"\nDistribution:")
    for key, val in metadata.get('distribution', {}).items():
        print(f"  {key}: {val}")
    print(f"\nShared keywords: {metadata.get('shared_keywords')}")
else:
    print("No metadata file found - run 02c_data_generator_v4.ipynb first!")

In [ ]:
# Cell 4: Create Label Mappings

INTENTS = [
    "order_status",
    "payment_info",
    "product_price",
    "product_stock",
    "product_description",
    "out_of_scope"
]

label2id = {intent: idx for idx, intent in enumerate(INTENTS)}
id2label = {idx: intent for idx, intent in enumerate(INTENTS)}

train_val_df['label'] = train_val_df['intent'].map(label2id)
test_ood_df['label'] = test_ood_df['intent'].map(label2id)

print("Label mappings:")
print(f"  label2id: {label2id}")

assert train_val_df['label'].isna().sum() == 0, "Some train intents couldn't be mapped!"
assert test_ood_df['label'].isna().sum() == 0, "Some test intents couldn't be mapped!"
print(f"\nAll samples mapped successfully!")

In [ ]:
# Cell 5: Train/Val Split

train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=train_val_df['label']
)

print("Dataset splits:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val: {len(val_df)} samples")
print(f"  Test (OOD): {len(test_ood_df)} samples")

In [ ]:
# Cell 6: Load Tokenizer and Model

MODEL_NAME = "cahya/distilbert-base-indonesian"

print(f"Loading model: {MODEL_NAME}")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded!")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENTS),
    id2label=id2label,
    label2id=label2id
)
print(f"Model loaded with {len(INTENTS)} labels!")

In [ ]:
# Cell 7: Tokenize Datasets

MAX_LENGTH = 128

def tokenize_data(df, tokenizer, max_length=MAX_LENGTH):
    df = df.reset_index(drop=True)
    
    encodings = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None
    )
    
    labels = [int(x) for x in df['label'].tolist()]
    
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })

print("Tokenizing datasets...")
train_dataset = tokenize_data(train_df, tokenizer)
val_dataset = tokenize_data(val_df, tokenizer)
test_ood_dataset = tokenize_data(test_ood_df, tokenizer)

print(f"Datasets tokenized!")
print(f"  Train: {len(train_dataset)}")
print(f"  Val: {len(val_dataset)}")
print(f"  Test (OOD): {len(test_ood_dataset)}")

In [ ]:
# Cell 8: Define Metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    if hasattr(logits, 'numpy'):
        logits = logits.numpy()
    logits = np.array(logits)
    
    if hasattr(labels, 'numpy'):
        labels = labels.numpy()
    labels = np.array(labels)
    
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Metrics function defined!")

In [ ]:
# Cell 9: Training Configuration
# Using longer training since semantic learning takes more epochs

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    
    # Training hyperparameters
    num_train_epochs=15,  # More epochs for semantic learning
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=200,  # More warmup
    
    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    
    # Logging
    logging_dir=str(MODEL_DIR / "logs"),
    logging_steps=50,
    report_to="none",
    
    # Other settings
    seed=42,
    fp16=torch.cuda.is_available(),
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")

In [ ]:
# Cell 10: Create Trainer

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # More patience
)

print("Trainer created with early stopping (patience=5)!")

In [ ]:
# Cell 11: Train Model

print("\n" + "=" * 60)
print("STARTING V4 TRAINING")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("IMPORTANT VALIDATION POINTS:")
print("  - Epoch 1 should be ~50-55% (if >70%, dataset has issues)")
print("  - Final should be ~85-90% (if >95%, dataset has issues)")
print()

train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 12: Validation Check

print("\n" + "=" * 60)
print("V4 VALIDATION CHECK")
print("=" * 60)

# Get training history
log_history = trainer.state.log_history
eval_logs = [log for log in log_history if 'eval_accuracy' in log]

if len(eval_logs) > 0:
    epoch_1_acc = eval_logs[0].get('eval_accuracy', 0)
    final_acc = eval_logs[-1].get('eval_accuracy', 0)
    
    print(f"\nEpoch 1 Accuracy: {epoch_1_acc:.2%}")
    print(f"Final Accuracy: {final_acc:.2%}")
    
    print("\nValidation:")
    
    # Check epoch 1
    if epoch_1_acc < 0.60:
        print(f"  [PASS] Epoch 1 accuracy ({epoch_1_acc:.2%}) < 60%")
        print(f"         This indicates the model is learning semantics, not keywords")
    else:
        print(f"  [WARN] Epoch 1 accuracy ({epoch_1_acc:.2%}) >= 60%")
        print(f"         Dataset may still have keyword exclusivity")
    
    # Check final
    if 0.80 <= final_acc <= 0.92:
        print(f"  [PASS] Final accuracy ({final_acc:.2%}) is in target range 80-92%")
        print(f"         This indicates true semantic learning")
    elif final_acc > 0.95:
        print(f"  [FAIL] Final accuracy ({final_acc:.2%}) > 95%")
        print(f"         Dataset still has keyword matching issues")
    else:
        print(f"  [INFO] Final accuracy ({final_acc:.2%})")
else:
    print("No evaluation logs found")

In [ ]:
# Cell 13: Save Best Model

BEST_MODEL_DIR = MODEL_DIR / "best_model"
BEST_MODEL_DIR.mkdir(exist_ok=True)

trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

label_mapping = {
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'intents': INTENTS,
    'confidence_threshold': 0.7,
    'version': 'v4'
}

with open(BEST_MODEL_DIR / "label_mapping.json", 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"Model saved to: {BEST_MODEL_DIR}")

In [ ]:
# Cell 14: Evaluate on OOD Test Set

print("\n" + "=" * 60)
print("OOD TEST SET EVALUATION")
print("=" * 60)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

all_preds = []
all_labels = []
all_confidences = []

for item in test_ood_dataset:
    input_ids = torch.tensor([item['input_ids']]).to(device)
    attention_mask = torch.tensor([item['attention_mask']]).to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred].item()
    
    all_preds.append(pred)
    all_labels.append(item['labels'])
    all_confidences.append(confidence)

ood_accuracy = np.mean([t == p for t, p in zip(all_labels, all_preds)])
print(f"\nOOD Test Accuracy: {ood_accuracy:.4f} ({ood_accuracy*100:.2f}%)")

In [ ]:
# Cell 15: Classification Report

print("\nClassification Report (OOD Test):")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=INTENTS, digits=4))

In [ ]:
# Cell 16: Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=INTENTS, yticklabels=INTENTS, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title(f'V4 Intent Classifier - OOD Confusion Matrix\nAccuracy: {ood_accuracy:.2%}', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix_v4.png', dpi=300)
plt.show()

In [ ]:
# Cell 17: Error Analysis by Sample Type

print("\n" + "=" * 60)
print("ERROR ANALYSIS BY SAMPLE TYPE")
print("=" * 60)

test_ood_df_reset = test_ood_df.reset_index(drop=True)

# Analyze by sample type
for sample_type in test_ood_df_reset['sample_type'].unique():
    mask = test_ood_df_reset['sample_type'] == sample_type
    type_indices = [i for i, m in enumerate(mask) if m]
    
    type_preds = [all_preds[i] for i in type_indices]
    type_labels = [all_labels[i] for i in type_indices]
    type_acc = sum(p == l for p, l in zip(type_preds, type_labels)) / len(type_preds)
    
    print(f"\n{sample_type}: {type_acc:.2%} ({sum(p == l for p, l in zip(type_preds, type_labels))}/{len(type_preds)})")

In [ ]:
# Cell 18: Save Results

correct_mask = [t == p for t, p in zip(all_labels, all_preds)]
correct_conf = [c for c, m in zip(all_confidences, correct_mask) if m]
wrong_conf = [c for c, m in zip(all_confidences, correct_mask) if not m]

training_summary = {
    'model_name': MODEL_NAME,
    'version': 'v4',
    'num_intents': len(INTENTS),
    'intents': INTENTS,
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples_ood': len(test_ood_df),
    'ood_test_accuracy': float(ood_accuracy),
    'epoch_1_accuracy': float(eval_logs[0].get('eval_accuracy', 0)) if eval_logs else None,
    'training_args': {
        'epochs': training_args.num_train_epochs,
        'learning_rate': training_args.learning_rate,
        'batch_size': training_args.per_device_train_batch_size,
    },
    'confidence_stats': {
        'correct_mean': float(np.mean(correct_conf)) if correct_conf else None,
        'wrong_mean': float(np.mean(wrong_conf)) if wrong_conf else None,
    },
    'v4_improvements': [
        'Shared keywords across intents (breaks exclusivity)',
        '28% generic patterns',
        '20% hard negatives',
        '10% ambiguous queries'
    ]
}

with open(MODEL_DIR / 'training_summary.json', 'w') as f:
    json.dump(training_summary, f, indent=2)
print(f"Training summary saved to: {MODEL_DIR / 'training_summary.json'}")

In [ ]:
# Cell 19: Quick Test

from transformers import pipeline

print("\n" + "=" * 60)
print("QUICK MODEL TEST")
print("=" * 60)

classifier = pipeline(
    "text-classification",
    model=str(BEST_MODEL_DIR),
    device=-1
)

# Test with shared keyword queries
test_queries = [
    # "berapa" tests (price vs stock vs order)
    ("harga berapa", "product_price"),
    ("stok berapa", "product_stock"),
    ("berapa lama sampai", "order_status"),
    
    # "ready" tests (stock vs payment)
    ("ready stock", "product_stock"),
    ("ready bayar", "payment_info"),
    
    # "gimana" tests (description vs order)
    ("produk gimana", "product_description"),
    ("pesanan gimana", "order_status"),
    
    # Generic queries
    ("mahal ga", "product_price"),
    ("kapan nyampe", "order_status"),
    ("pake apa", "payment_info"),
]

print("\nTesting semantic understanding:")
print("-" * 60)

correct = 0
for query, expected in test_queries:
    result = classifier(query)[0]
    is_correct = result['label'] == expected
    correct += is_correct
    status = "OK" if is_correct else "WRONG"
    print(f"\n\"{query}\"")
    print(f"  Expected: {expected}")
    print(f"  Got: {result['label']} (conf: {result['score']:.3f}) [{status}]")

print(f"\nQuick test accuracy: {correct}/{len(test_queries)} ({correct/len(test_queries)*100:.0f}%)")

In [ ]:
# Cell 20: Summary

print("\n" + "=" * 60)
print("V4 TRAINING COMPLETE - SUMMARY")
print("=" * 60)

epoch_1_acc = eval_logs[0].get('eval_accuracy', 0) if eval_logs else 0

print(f"""
Model: {MODEL_NAME}
Version: V4 (Semantic Learning)

Training Curve Validation:
--------------------------
Epoch 1 Accuracy: {epoch_1_acc:.2%} (target: 50-55%)
Final Accuracy: {ood_accuracy:.2%} (target: 85-90%)

Comparison:
-----------
V3 Epoch 1: ~84% → V4 Epoch 1: {epoch_1_acc:.2%}
V3 Final: ~98% → V4 Final: {ood_accuracy:.2%}

{'SUCCESS: Model is learning semantics!' if epoch_1_acc < 0.65 and 0.80 <= ood_accuracy <= 0.92 else 'Check dataset if accuracy is off-target'}

Key V4 Changes:
---------------
- Shared keywords (berapa, ready, ada, gimana, info, bisa)
- 28% generic patterns (no exclusive keywords)
- 20% hard negatives (confusing pairs)
- 10% ambiguous queries

Files saved:
  - {BEST_MODEL_DIR}
  - {MODEL_DIR / 'training_summary.json'}
  - {MODEL_DIR / 'confusion_matrix_v4.png'}
""")